In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model_name = "google/gemma-2-2b"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="mps",
                                             dtype=torch.float16,
                                            )
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,  # replace with "mps" to run on a Mac device
# )

# text = "Once upon a time,"
# outputs = pipe(text, max_new_tokens=256)
# response = outputs[0]["generated_text"]
# print(response)


/Users/souvikb/miniforge3/envs/SPAR_clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 288/288 [00:06<00:00, 41.69it/s]


In [3]:
# Loading the GSM8K dataset and preparing it for fine-tuning
train_ds = load_dataset("openai/gsm8k", "main", split="train[:200]")
def format_example(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"

# train_ds = train_ds.map(format_example)

In [4]:
# Configuring LoRA for fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05,
)
# unload the model to free up memory before applying LoRA
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="mps", dtype=torch.float16)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


In [5]:
# Training configuration
training_args = SFTConfig(
    num_train_epochs=1, 
    per_device_train_batch_size=16,
    logging_steps=10,
    output_dir="./gemma-gsm8k-lora-all",
    learning_rate=2e-4,
    fp16=True,
    save_strategy='epoch',
    max_length=512,
    gradient_accumulation_steps=1,
    report_to="none",
)

In [6]:
# Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    formatting_func=format_example,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset: 100%|██████████| 200/200 [00:00<00:00, 115418.38 examples/s]


In [7]:
print("Starting training...")
trainer.train()

print("Saving model...")
trainer.save_model("./gemma-gsm8k-lora-all-final")

print("Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Starting training...


/Users/souvikb/miniforge3/envs/SPAR_clean/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.168540


Saving model...
Training complete!


# LoRA on late layers only

In [11]:
## Print the number of hidden layers of the model
print(model.config.num_hidden_layers)

26


In [12]:
# Using a fresh base model to make sure that we do not 
# run LoRA on top of already LoRA run model
base_id = "google/gemma-2-2b"

def load_base():
    return AutoModelForCausalLM.from_pretrained(base_id,dtype=torch.float16, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(base_id)
tokenizer.pad_token = tokenizer.eos_token
# 2. Configure late_LoRA

lora_config_late = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    layers_to_transform=list(range(13, 26)), #layers 13--25
    layers_pattern="layers",
)
model_late = load_base()
model_late = get_peft_model(model_late, lora_config_late)
print("\n=== LATE LAYERS ONLY ===")
model_late.print_trainable_parameters() #check how drammatically this reduces the number of parameters. 

Loading weights: 100%|██████████| 288/288 [00:12<00:00, 23.89it/s]



=== LATE LAYERS ONLY ===
trainable params: 1,597,440 || all params: 2,615,939,328 || trainable%: 0.0611


In [13]:
# 4. Training config optimized for M4 Max
training_args = SFTConfig(
    output_dir="./gemma-gsm8k-lora-late",
    num_train_epochs=1,
    per_device_train_batch_size=16,  # M4 Max can handle larger batches
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,  # Use FP16 for speed
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    report_to="none",
)

In [14]:
# 5. Train
trainer = SFTTrainer(
    model=model_late,
    args=training_args,
    train_dataset=train_ds,
    formatting_func=format_example,
    processing_class=tokenizer,
)

In [15]:
print("\nStarting training (late layers only)...")
trainer.train()

print("Saving model...")
trainer.save_model("./gemma-gsm8k-lora-late-final")

print("Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.



Starting training (late layers only)...


/Users/souvikb/miniforge3/envs/SPAR_clean/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.253967


Saving model...
Training complete!


In [20]:
## Comparing the two fine-tuning:

import re
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

base_id = "google/gemma-2-2b"
all_adapter_path = "./gemma-gsm8k-lora-all-final"
late_adapter_path = "./gemma-gsm8k-lora-late-final"

eval_ds = load_dataset("openai/gsm8k", "main", split="test[:100]")
tokenizer = AutoTokenizer.from_pretrained(base_id)
tokenizer.pad_token = tokenizer.eos_token

def load_base():
    return AutoModelForCausalLM.from_pretrained(
        base_id,
        dtype=torch.float16,
        device_map="auto",
    )

def load_adapter(adapter_path):
    base_model = load_base()
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model

def build_prompt(example):
    return f"Question: {example['question']}\nAnswer:"

def extract_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", answer_text)
    if match:
        return match.group(1).replace(",", "")
    return None

def extract_pred_answer(text):
    matches = re.findall(r"-?[\d,]+(?:\.\d+)?", text)
    if not matches:
        return None
    return matches[-1].replace(",", "")

@torch.no_grad()
def generate_response(model, question, max_new_tokens=128):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()
    return decoded

def evaluate_model(model, dataset):
    results = []
    correct = 0

    for ex in dataset:
        pred_text = generate_response(model, ex["question"])
        pred_answer = extract_pred_answer(pred_text)
        gold_answer = extract_gold_answer(ex["answer"])
        is_correct = pred_answer == gold_answer

        correct += int(is_correct)
        results.append({
            "question": ex["question"],
            "gold": gold_answer,
            "pred": pred_answer,
            "correct": is_correct,
            "raw_output": pred_text,
        })

    accuracy = correct / len(results)
    return accuracy, results

model_all = load_adapter(all_adapter_path)
acc_all, results_all = evaluate_model(model_all, eval_ds)
print("Full LoRA accuracy:", acc_all)

del model_all
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

model_late = load_adapter(late_adapter_path)
acc_late, results_late = evaluate_model(model_late, eval_ds)
print("Late LoRA accuracy:", acc_late)




Loading weights: 100%|██████████| 288/288 [00:19<00:00, 14.94it/s]


Full LoRA accuracy: 0.21


Loading weights: 100%|██████████| 288/288 [00:03<00:00, 87.67it/s] 


Late LoRA accuracy: 0.03
